In [3]:
%cd /content
!git clone https://github.com/Genpactal/CodeGuardLLM.git
%cd /content/CodeGuardLLM
!ls scripts/robustness.py

/content
Cloning into 'CodeGuardLLM'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 147 (delta 63), reused 126 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 972.19 KiB | 3.10 MiB/s, done.
Resolving deltas: 100% (63/63), done.
/content/CodeGuardLLM
scripts/robustness.py


# Code watermark robustness

Compare fixed code transformations with an unchanged baseline on saved SWEET or KGW runs. Both clean and watermarked solutions are attacked. This notebook does not regenerate solutions or change the core experiment results.

The saved SWEET run below is the one already evaluated locally. Change `RUN_ROOT` for a different experiment. Scoring and correctness are separate opt-in steps; until both finish the robustness report is explicitly pending.


In [1]:
from pathlib import Path
import json, os, subprocess, sys
from datetime import datetime, timezone
from IPython.display import Image, Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/robustness.py').is_file())
RUN_ROOT = ROOT / 'runs/sweet-import/0eedc6730e049860/sweet_comparison'
RUN_DIRS = [RUN_ROOT / f'delta-{delta}' for delta in (1, 2, 3)]
PYTHON = next((str(p) for p in (ROOT / '.venv-colab/bin/python', ROOT / '.venv-gpu/Scripts/python.exe', ROOT / '.venv/Scripts/python.exe') if p.is_file()), sys.executable)
RUN_SCORING = True
RUN_CORRECTNESS = True
SCORE_OPTIONS = []  # For deliberate CPU scoring: ['--device', 'cpu', '--dtype', 'float32']
for directory in RUN_DIRS:
    if not (directory / 'manifest.json').is_file():
        raise FileNotFoundError(f'Point RUN_ROOT to your saved experiment: {directory}')
print('Interpreter:', PYTHON)
print('Source runs:', RUN_ROOT)


StopIteration: 

In [ ]:
LOG_DIR = ROOT / '.tmp/robustness-logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

def run(action, directory, *options):
    log = LOG_DIR / f"{directory.name}-{action}-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')}.log"
    args = [PYTHON, '-u', 'scripts/robustness.py', action, '--run-dir', str(directory), *options]
    print(f'{action}: {directory.name}; log: {log}', flush=True)
    with log.open('w', encoding='utf-8') as handle:
        with subprocess.Popen(args, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace',
                              env={**os.environ, 'MPLBACKEND': 'Agg'}) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    handle.write(line)
                    handle.flush()
                status = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if status:
        raise RuntimeError(f'{action} failed (exit {status}); full log: {log}')


## 1. Prepare frozen variants

Identity, comment removal, AST reformatting, docstring removal, and conservative local-variable renaming. AST edits also normalize formatting, so use the reformat arm as a control. No-op edits and failures remain in the report. Source code is parsed here, never executed.


In [ ]:
for directory in RUN_DIRS:
    run('prepare', directory)


prepare: delta-1; log: c:\Users\user\github\CodeGuardLLM\.tmp\robustness-logs\delta-1-prepare-20261018-125541-286745.log
<unknown>:5: SyntaxWarning: invalid escape sequence '\d'
<robustness-candidate>:5: SyntaxWarning: invalid escape sequence '\d'
<unknown>:5: SyntaxWarning: invalid escape sequence '\d'
<unknown>:5: SyntaxWarning: invalid escape sequence '\d'
<unknown>:5: SyntaxWarning: invalid escape sequence '\d'
<unknown>:5: SyntaxWarning: invalid escape sequence '\d'
<unknown>:5: SyntaxWarning: invalid escape sequence '\d'
<robustness-candidate>:5: SyntaxWarning: invalid escape sequence '\d'
<unknown>:5: SyntaxWarning: invalid escape sequence '\d'
Prepared 1640 variants in C:\Users\user\github\CodeGuardLLM\runs\sweet-import\0eedc6730e049860\sweet_comparison\delta-1\robustness
prepare: delta-2; log: c:\Users\user\github\CodeGuardLLM\.tmp\robustness-logs\delta-2-prepare-20261018-125541-847254.log
<unknown>:5: SyntaxWarning: invalid escape sequence '\d'
<unknown>:9: SyntaxWarning: inv

## 2. Rescore identity and attacks

Set `RUN_SCORING=True` above to run. SWEET needs the model to calculate entropy. All variants, including identity, use the same detector environment; old generation scores are not substituted. To run this stage in Colab, transfer the project and full source run including the frozen `robustness` folders, select its project environment, and adjust `RUN_ROOT`. Copy the scored folders back for local Docker evaluation. Completed rows are reused on resume.


In [ ]:
if RUN_SCORING:
    for directory in RUN_DIRS:
        run('score', directory, *SCORE_OPTIONS)
else:
    print('Scoring not requested. Existing saved scores can still be analyzed.')


Scoring not requested. Existing saved scores can still be analyzed.


## 3. Test correctness in Docker

Start Docker Desktop and set `RUN_CORRECTNESS=True` above. The evaluator validates canonical reference solutions first, then tests the frozen variants with the source run's limits. Correctness alone does not establish evasion; detection scores are also needed.


In [ ]:
if RUN_CORRECTNESS:
    for directory in RUN_DIRS:
        run('evaluate', directory)
else:
    print('Correctness evaluation not requested. Missing results remain pending.')


Correctness evaluation not requested. Missing results remain pending.


## 4. Robustness results

A correctness-preserving evasion needs a baseline that passed and was detected, a changed candidate that still passes, and a valid attacked score below or equal to the unchanged threshold. Unscored code is not a successful evasion. No eligible baselines makes the evasion rate unavailable. These fixed attacks do not measure adaptive rewriting or prove semantic equivalence.


In [ ]:
for directory in RUN_DIRS:
    run('analyze', directory)
    output = directory / 'robustness'
    summary = json.loads((output / 'robustness_summary.json').read_text(encoding='utf-8'))
    display(Markdown(f"### {directory.name}: {summary['status']}"))
    text = (output / 'report.md').read_text(encoding='utf-8')
    display(Markdown(text.replace('![Detection and correctness after transformations](robustness.png)', '')))
    if (output / 'robustness.png').is_file():
        display(Image(filename=str(output / 'robustness.png')))


analyze: delta-1; log: c:\Users\user\github\CodeGuardLLM\.tmp\robustness-logs\delta-1-analyze-20261018-125542-987463.log
Robustness report: c:\Users\user\github\CodeGuardLLM\runs\sweet-import\0eedc6730e049860\sweet_comparison\delta-1\robustness\report.md (pending)


### delta-1: pending

# Code watermark robustness

Status: **pending**; fixed Z > 4.

Identity and attacks are scored in the same detector environment. Clean code receives the same attacks. AST attacks also normalize formatting and remove comments; compare them with reformat, not only identity.

| Attack | Condition | Changed / total | Scored | Detected rate | AUROC | Pass@1 | Correctness retention | Correct evasions / eligible |
|---|---|---|---|---|---|---|---|---|
| identity | clean | 0/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| identity | watermarked | 0/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| strip_comments | clean | 17/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| strip_comments | watermarked | 26/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| reformat | clean | 161/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| reformat | watermarked | 162/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| strip_docstrings | clean | 25/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| strip_docstrings | watermarked | 18/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| rename_locals | clean | 70/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| rename_locals | watermarked | 69/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |

Pass@1 and detected rate include every source sample, including failed transformations. AUROC excludes unscored code. Missing stages remain pending. Unscored code is not a confirmed evasion. Correct evasions require a changed candidate, passing baseline and attacked code, baseline detection, and a valid attacked score at or below the fixed threshold. No eligible samples makes this rate unavailable.

Correctness retention is the fraction of baseline-passing solutions that still pass. Detection retention includes unscored attacked outputs as not retained. Small detected subsets are not reliable estimates. Test passing is not proof of semantic equivalence. These fixed transformations do not measure adaptive or LLM-assisted attacks.


analyze: delta-2; log: c:\Users\user\github\CodeGuardLLM\.tmp\robustness-logs\delta-2-analyze-20261018-125543-121702.log
Robustness report: c:\Users\user\github\CodeGuardLLM\runs\sweet-import\0eedc6730e049860\sweet_comparison\delta-2\robustness\report.md (pending)


### delta-2: pending

# Code watermark robustness

Status: **pending**; fixed Z > 4.

Identity and attacks are scored in the same detector environment. Clean code receives the same attacks. AST attacks also normalize formatting and remove comments; compare them with reformat, not only identity.

| Attack | Condition | Changed / total | Scored | Detected rate | AUROC | Pass@1 | Correctness retention | Correct evasions / eligible |
|---|---|---|---|---|---|---|---|---|
| identity | clean | 0/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| identity | watermarked | 0/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| strip_comments | clean | 17/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| strip_comments | watermarked | 43/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| reformat | clean | 161/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| reformat | watermarked | 160/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| strip_docstrings | clean | 25/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| strip_docstrings | watermarked | 16/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| rename_locals | clean | 70/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| rename_locals | watermarked | 66/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |

Pass@1 and detected rate include every source sample, including failed transformations. AUROC excludes unscored code. Missing stages remain pending. Unscored code is not a confirmed evasion. Correct evasions require a changed candidate, passing baseline and attacked code, baseline detection, and a valid attacked score at or below the fixed threshold. No eligible samples makes this rate unavailable.

Correctness retention is the fraction of baseline-passing solutions that still pass. Detection retention includes unscored attacked outputs as not retained. Small detected subsets are not reliable estimates. Test passing is not proof of semantic equivalence. These fixed transformations do not measure adaptive or LLM-assisted attacks.


analyze: delta-3; log: c:\Users\user\github\CodeGuardLLM\.tmp\robustness-logs\delta-3-analyze-20261018-125543-271856.log
Robustness report: c:\Users\user\github\CodeGuardLLM\runs\sweet-import\0eedc6730e049860\sweet_comparison\delta-3\robustness\report.md (pending)


### delta-3: pending

# Code watermark robustness

Status: **pending**; fixed Z > 4.

Identity and attacks are scored in the same detector environment. Clean code receives the same attacks. AST attacks also normalize formatting and remove comments; compare them with reformat, not only identity.

| Attack | Condition | Changed / total | Scored | Detected rate | AUROC | Pass@1 | Correctness retention | Correct evasions / eligible |
|---|---|---|---|---|---|---|---|---|
| identity | clean | 0/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| identity | watermarked | 0/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| strip_comments | clean | 17/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| strip_comments | watermarked | 58/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| reformat | clean | 161/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| reformat | watermarked | 163/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| strip_docstrings | clean | 25/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| strip_docstrings | watermarked | 16/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| rename_locals | clean | 70/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |
| rename_locals | watermarked | 65/164 | 0 | N/A or pending | N/A or pending | N/A or pending | N/A or pending | pending |

Pass@1 and detected rate include every source sample, including failed transformations. AUROC excludes unscored code. Missing stages remain pending. Unscored code is not a confirmed evasion. Correct evasions require a changed candidate, passing baseline and attacked code, baseline detection, and a valid attacked score at or below the fixed threshold. No eligible samples makes this rate unavailable.

Correctness retention is the fraction of baseline-passing solutions that still pass. Detection retention includes unscored attacked outputs as not retained. Small detected subsets are not reliable estimates. Test passing is not proof of semantic equivalence. These fixed transformations do not measure adaptive or LLM-assisted attacks.
